# Tool useとAIエージェント


In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## Function calling


### Function callingをそのまま使う例


In [ ]:
import json


def get_current_weather(location: str, unit: str = "celsius") -> str:
    return json.dumps({"location": location, "temperature": 20, "unit": unit})

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city, e.g. Tokyo",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "default": "celsius",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "東京の現在の天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages,
    tools=tools,
    reasoning_effort="none",
)
print(response.to_json(indent=2))

In [ ]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())
print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)
    print(function_response.encode("utf-8").decode("unicode_escape"))

    # 関数の実行結果を会話履歴としてmessagesに追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
second_response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages,
    reasoning_effort="none",
)
print(second_response.to_json(indent=2))

## エージェントループの実装

### Chat Completions APIでエージェントループを実装する

In [ ]:
import inspect
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

from pydantic import create_model


@dataclass
class Tool:
    """Chat Completions APIに渡す定義と、実際に呼び出す関数の組"""

    definition: dict[str, Any]
    function: Callable[..., str]


def function_to_tool(func: Callable[..., str]) -> Tool:
    """関数の型ヒントとdocstringから、Chat Completions APIに渡すtoolの定義を作る"""
    fields: dict[str, Any] = {}
    for name, param in inspect.signature(func).parameters.items():
        default = ... if param.default is inspect.Parameter.empty else param.default
        fields[name] = (param.annotation, default)
    parameters = create_model(func.__name__, **fields).model_json_schema()
    definition = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": inspect.getdoc(func) or "",
            "parameters": parameters,
        },
    }
    return Tool(definition=definition, function=func)

In [ ]:
from typing import Literal


def get_current_weather(
    location: str, unit: Literal["celsius", "fahrenheit"] = "celsius"
) -> str:
    """Get the current weather in a given location"""
    return json.dumps({"location": location, "temperature": 20, "unit": unit})


tools = [function_to_tool(get_current_weather)]
print(json.dumps(tools[0].definition, indent=2))

In [ ]:
def agent_loop(
    messages: list,
    tools: list[Tool],
    max_iterations: int = 20,
) -> str | None:
    """LLMがツールを使いたいと応答する限り、ツールを実行して結果を渡し続ける（エージェントループ）"""
    available_functions = {tool.definition["function"]["name"]: tool.function for tool in tools}

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages,
            tools=[tool.definition for tool in tools],
            reasoning_effort="none",
        )
        response_message = response.choices[0].message
        messages.append(response_message.to_dict())

        # ツールを使わない応答なら、それが最終的な回答
        if not response_message.tool_calls:
            return response_message.content

        # ツールを使いたいという応答なら、ツールを実行して結果をmessagesに追加し、再度LLMを呼び出す
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"[tool] {function_name}({function_args})")
            function_response = available_functions[function_name](**function_args)
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )

    return "（反復回数の上限に達しました）"

In [ ]:
messages = [
    {"role": "user", "content": "東京と大阪の現在の天気を比べて教えてください"},
]

answer = agent_loop(messages, tools)
print(answer)

### LangChainのcreate_agentと@toolでの実装


In [ ]:
from typing import Any
from langchain.tools import tool


@tool
def get_current_weather(location: str, unit: str = "celsius") -> dict[str, Any]:
    """Get the current weather in a given location"""
    return {"location": location, "temperature": 20, "unit": unit}

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5.6-luna",
    model_provider="openai",
    reasoning_effort="none",
)


agent = create_agent(
    model=model,
    tools=[get_current_weather],
)

In [ ]:
from langchain.messages import HumanMessage

initial_state = {"messages": [HumanMessage(content="東京の現在の天気はどうですか？")]}

for event in agent.stream(initial_state, stream_mode="updates"):
    for value in event.values():
        latest_message = value["messages"][-1]
        latest_message.pretty_print()
